In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import hf_hub_download
import json

In [ ]:
print("Loading METR-LA dataset...")
train = load_dataset("witgaw/METR-LA", split="train")

In [ ]:
CURRENT_SPEED = "x_t+0_d0" if "x_t+0_d0" in train.column_names else "x_t-0_d0"

In [ ]:
print("Extracting traffic data...")
traffic = train.select_columns(["node_id", "t0_timestamp", CURRENT_SPEED, "y_t+1_d0"]).to_pandas()
traffic = traffic.rename(columns={CURRENT_SPEED: "speed", "y_t+1_d0": "next_speed"})
traffic["node_id"] = traffic["node_id"].astype(int)
traffic["t0_timestamp"] = pd.to_datetime(traffic["t0_timestamp"])
traffic = traffic.sort_values(["node_id", "t0_timestamp"]).reset_index(drop=True)

In [ ]:
print("Computing zero_rate...")
traffic["near_zero"] = traffic["speed"] <= 1.0
failure_df = traffic.groupby("node_id", as_index=False).agg(
    zero_rate=("near_zero", "mean"),
    observations=("speed", "size"),
    avg_speed=("speed", "mean")
)

In [ ]:
# traffic regime based on average speed
failure_df["traffic_regime"] = np.where(failure_df["avg_speed"] < 40, "congested", "free_flow")
failure_df["road_type"] = "highway" # placeholder for all sensors in this dataset

In [ ]:
print("Computing CUSUM and EWMA rates...")
def detect_cusum(values, threshold=5.0, drift=0.5):
    values = np.asarray(values, dtype=float)
    mean = np.mean(values)
    std = np.std(values)
    if std == 0: return 0
    z = (values - mean) / std
    positive = 0.0
    negative = 0.0
    flags = 0
    for value in z:
        positive = max(0, positive + value - drift)
        negative = min(0, negative + value + drift)
        if positive > threshold or negative < -threshold:
            flags += 1
            positive = 0.0
            negative = 0.0
    return flags

In [ ]:
def detect_ewma(values, alpha=0.2, control_limit=3.0):
    values = np.asarray(values, dtype=float)
    mean = np.mean(values)
    std = np.std(values)
    if std == 0: return 0
    ewma = mean
    flags = 0
    ewma_std = std * np.sqrt(alpha / (2 - alpha))
    upper = mean + control_limit * ewma_std
    lower = mean - control_limit * ewma_std
    for value in values:
        ewma = alpha * value + (1 - alpha) * ewma
        if ewma > upper or ewma < lower:
            flags += 1
    return flags

In [ ]:
drift_results = []
grouped = traffic.groupby("node_id", sort=True)
for node_id, group in grouped:
    group = group.sort_values("t0_timestamp")
    values = group["speed"].to_numpy(dtype=float)
    cusum_flags = detect_cusum(values)
    ewma_flags = detect_ewma(values)
    
    # persistence error
    pers_err = np.mean(np.abs(group["speed"] - group["next_speed"]))
    
    drift_results.append({
        "node_id": int(node_id),
        "cusum_flags": cusum_flags,
        "ewma_flags": ewma_flags,
        "persistence_error": pers_err
    })

In [ ]:
drift_df = pd.DataFrame(drift_results)
metrics_df = failure_df.merge(drift_df, on="node_id")

In [ ]:
metrics_df["cusum_flag_rate"] = metrics_df["cusum_flags"] / metrics_df["observations"]
metrics_df["ewma_flag_rate"] = metrics_df["ewma_flags"] / metrics_df["observations"]

In [ ]:
print("Downloading graph files for density and topology...")
REPO_ID = "witgaw/METR-LA"
adj_path = hf_hub_download(repo_id=REPO_ID, filename="sensor_graph/adj_mx.npy", repo_type="dataset")
loc_path = hf_hub_download(repo_id=REPO_ID, filename="sensor_graph/sensor_locations.csv", repo_type="dataset")
mapping_path = hf_hub_download(repo_id=REPO_ID, filename="sensor_graph/adj_mx_mapping.json", repo_type="dataset")

In [ ]:
adj_mx = np.load(adj_path, allow_pickle=False)
locations = pd.read_csv(loc_path)
with open(mapping_path, "r") as f:
    mapping = json.load(f)

In [ ]:
# The row indices of adj_mx correspond to node_ids 0..206 directly as seen in 01_06_summary.md (node_id ↔ adjacency alignment passed).
# So we can just sum the adjacency matrix rows for topology.
# topology: sum of out-weights
metrics_df["topology"] = adj_mx.sum(axis=1)

In [ ]:
# compute geographic density (neighbors within 1km)
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000 # radius of Earth in meters
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    delta_phi = np.radians(lat2 - lat1)
    delta_lambda = np.radians(lon2 - lon1)
    a = np.sin(delta_phi / 2.0) ** 2 + np.cos(phi1) * np.cos(phi2) * np.sin(delta_lambda / 2.0) ** 2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))
    return R * c

In [ ]:
print("Computing density...")
# get physical sensor_ids for node_id 0..206
sensor_ids_ordered = mapping["sensor_ids"]

In [ ]:
density_scores = []
for i, s_id in enumerate(sensor_ids_ordered):
    # node i corresponds to sensor_id s_id
    s_id_int = int(s_id)
    if s_id_int in locations["sensor_id"].values:
        row = locations[locations["sensor_id"] == s_id_int].iloc[0]
        lat1, lon1 = row["latitude"], row["longitude"]
        neighbors = 0
        for j, other_id in enumerate(sensor_ids_ordered):
            other_id_int = int(other_id)
            if i != j and other_id_int in locations["sensor_id"].values:
                row2 = locations[locations["sensor_id"] == other_id_int].iloc[0]
                lat2, lon2 = row2["latitude"], row2["longitude"]
                dist = haversine(lat1, lon1, lat2, lon2)
                if dist <= 1000.0:
                    neighbors += 1
        density_scores.append(neighbors)
    else:
        density_scores.append(0)

In [ ]:
metrics_df["density"] = density_scores

In [ ]:
print("Saving metr_la_metrics.csv...")
metrics_df.to_csv("metr_la_metrics.csv", index=False)
print("Done. Generated metr_la_metrics.csv with columns:")
print(metrics_df.columns.tolist())